<a href="https://colab.research.google.com/github/farida596/Medical-Chatbot-QLoRA/blob/main/Copy_of_medical_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y transformers trl peft accelerate bitsandbytes torch torchvision torchaudio

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [ ]:
!pip install -q \
torch==2.5.1 \
transformers==4.56.1 \
trl==0.21.0 \
peft==0.17.1 \
accelerate==1.10.1 \
bitsandbytes==0.47.0 \
datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 825.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,PeftModel
from trl import SFTTrainer,SFTConfig

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
ds=load_dataset('lavita/ChatDoctor-HealthCareMagic-100k',split='train').select(range(2000))
print(ds.column_names)
print(ds[0])

['instruction', 'input', 'output']
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I woke up this morning feeling the whole room is spinning when i was sitting down. I went to the bathroom walking unsteadily, as i tried to focus i feel nauseous. I try to vomit but it wont come out.. After taking panadol and sleep for few hours, i still feel the same.. By the way, if i lay down or sit down, my head do not spin, only when i want to move around then i feel the whole world is spinning.. And it is normal stomach discomfort at the same time? Earlier after i relieved myself, the spinning lessen so i am not sure whether its connected or coincidences.. Thank you doc!', 'output': 'Hi, Thank you for posting your query. The most likely cause for your symptoms is benign paroxysmal positional vertigo (BPPV), a type of peripheral vertigo. In this condition, the most common symptom is dizziness or giddiness, which is made worse 

In [ ]:
def format_example(x):
 user=x.get('input') or x.get('instruction') or x.get('question')
 assistant=x.get('output') or x.get('response') or x.get('answer')
 return {'text':tokenizer.apply_chat_template([{'role':'user','content':user},{'role':'assistant','content':assistant}],tokenize=False,add_generation_prompt=False)}
ds=ds.map(format_example,remove_columns=ds.column_names)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
peft=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
args = SFTConfig(
    output_dir="./medical-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    bf16=False,
    packing=False,
    max_length=256,
)
trainer=SFTTrainer(model=model,args=args,train_dataset=ds,processing_class=tokenizer,peft_config=peft)
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
100,2.351100
200,2.241000
300,2.147300
400,2.106100
500,2.108800


TrainOutput(global_step=500, training_loss=2.190858734130859, metrics={'train_runtime': 2446.4133, 'train_samples_per_second': 1.635, 'train_steps_per_second': 0.204, 'total_flos': 1.5473200107208704e+16, 'train_loss': 2.190858734130859})

In [ ]:
trainer.model.save_pretrained('medical-lora')
tokenizer.save_pretrained('medical-lora')

('medical-lora/tokenizer_config.json',
 'medical-lora/special_tokens_map.json',
 'medical-lora/chat_template.jinja',
 'medical-lora/vocab.json',
 'medical-lora/merges.txt',
 'medical-lora/added_tokens.json',
 'medical-lora/tokenizer.json')

In [ ]:
base=AutoModelForCausalLM.from_pretrained(model_name,quantization_config=bnb,device_map='auto')
model=PeftModel.from_pretrained(base,'medical-lora')
model.eval()
messages=[{'role':'user','content':'What are the symptoms of diabetes?'}]
text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
inputs=tokenizer(text,return_tensors='pt').to(model.device)
out=model.generate(**inputs,max_new_tokens=200,temperature=0.7)
print(tokenizer.decode(out[0],skip_special_tokens=True))

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What are the symptoms of diabetes?
assistant
Hello and welcome to Chat Doctor. As a diabetics specialist, i can understand your concern. Diabetes is a chronic disease which is characterized by high blood sugar levels. It can cause many complications in the body. The common symptoms of diabetes are
